**Bank statement classification:**

In [1]:
import torch
print(torch.cuda.is_available())

True


In [2]:
# ============================================
# STEP 0. Install packages and fix environment
# ============================================

!pip install -q transformers datasets accelerate evaluate peft openpyxl
!pip uninstall -y torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.9 MB/s eta 0:00:00
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU NOT FOUND")

CUDA available: True
GPU: Tesla T4


In [4]:
# import os
# os.kill(os.getpid(), 9)

In [5]:
import pandas as pd
import numpy as np
import torch

from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
file_path = "/content/drive/MyDrive/BS_train.xlsx"

df = pd.read_excel(file_path)

print("Dataset shape:", df.shape)
display(df.head())
print(df.columns.tolist())

Dataset shape: (8807, 16)


,Organisation,CashAccount,Counterparty,Gr1_CE,CostElement,Description1,Description2,Description3,Description4,Description5,Date,CurrencyOriginal,Amount_Curr,Amount_EUR,Субконто1 Дт,Субконто1 Кт
0,MedPrestige,Capitalist,CURRENCY CONVERSION 599.05 USD TO 550.00 EUR,Прочие доходы/расходы,Финансовые доходы/расходы,TRANSFER,637822242,CONVERSION,E14684723,NaN,2024-04-05,EUR,1.00001,1.00001,NaN,NaN
1,MedPrestige,Capitalist,PAYMENT,Инвестиции,Инвестиции,TRANSFER,637822473,OUT,E15001481,NaN,2024-04-05,EUR,1.00001,1.00001,NaN,NaN
2,MedPrestige,Capitalist,PAYMENT,Инвестиции,Инвестиции,TRANSFER,637822814,OUT,E15002766,NaN,2024-04-05,EUR,1.00001,1.00001,NaN,NaN
3,MedPrestige,Capitalist,CURRENCY CONVERSION 331.74 USD TO 307.00 EUR,Прочие доходы/расходы,Финансовые доходы/расходы,TRANSFER,639355147,CONVERSION,E14684723,NaN,2024-04-10,EUR,1.00001,1.00001,NaN,NaN
4,MedPrestige,Capitalist,PAYMENT,Инвестиции,Инвестиции,TRANSFER,639356258,OUT,E15002766,NaN,2024-04-10,EUR,1.00001,1.00001,NaN,NaN


['Organisation', 'CashAccount', 'Counterparty', 'Gr1_CE', 'CostElement', 'Description1', 'Description2', 'Description3', 'Description4', 'Description5', 'Date', 'CurrencyOriginal', 'Amount_Curr', 'Amount_EUR', 'Субконто1 Дт', 'Субконто1 Кт']


In [7]:
# STEP 2. Select columns

input_cols = ["CashAccount", "Counterparty", "Description1"]
target_col = "CostElement"

df_model = df[input_cols + [target_col]].copy()

print("Shape after selecting columns:", df_model.shape)
display(df_model.head())

Shape after selecting columns: (8807, 4)


,CashAccount,Counterparty,Description1,CostElement
0,Capitalist,CURRENCY CONVERSION 599.05 USD TO 550.00 EUR,TRANSFER,Финансовые доходы/расходы
1,Capitalist,PAYMENT,TRANSFER,Инвестиции
2,Capitalist,PAYMENT,TRANSFER,Инвестиции
3,Capitalist,CURRENCY CONVERSION 331.74 USD TO 307.00 EUR,TRANSFER,Финансовые доходы/расходы
4,Capitalist,PAYMENT,TRANSFER,Инвестиции


In [8]:
# STEP 3. Clean missing values

# Fill missing values in input text columns
# BERT cannot process NaN values, so we replace them with empty strings
for col in input_cols:
    df_model[col] = df_model[col].fillna("").astype(str)

# Remove rows where the target label is missing
# We cannot train the model without the correct CostElement
df_model = df_model.dropna(subset=[target_col])

# Convert target labels to string format
df_model[target_col] = df_model[target_col].astype(str)

# Show the dataset size after cleaning
print("Shape after cleaning:", df_model.shape)

# Check if any missing values remain
print("\nMissing values after cleaning:")
print(df_model.isna().sum())

Shape after cleaning: (8807, 4)

Missing values after cleaning:
CashAccount     0
Counterparty    0
Description1    0
CostElement     0
dtype: int64


In [9]:
# STEP 3. Define column mapping

COL_COUNTERPARTY = "Counterparty"
COL_DESCRIPTION = "Description1"
COL_ACCOUNT = "CashAccount"

In [10]:
# Clean text

import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [11]:
# Build text

def build_text(row):
    counterparty = clean_text(str(row[COL_COUNTERPARTY]))
    description = clean_text(str(row[COL_DESCRIPTION]))
    account = clean_text(str(row[COL_ACCOUNT]))

    # Neutral naming → works for any dataset
    text = (
        f"{counterparty}. "
        f"type: {description}. "
        f"account: {account}."
    )

    return text


df_model["text"] = df_model.apply(build_text, axis=1)

display(df_model[["text", target_col]].head(10))

,text,CostElement
0,currency conversion . usd to . eur. type: tran...,Финансовые доходы/расходы
1,payment. type: transfer. account: capitalist.,Инвестиции
2,payment. type: transfer. account: capitalist.,Инвестиции
3,currency conversion . usd to . eur. type: tran...,Финансовые доходы/расходы
4,payment. type: transfer. account: capitalist.,Инвестиции
5,inv. # rgp- -med. type: transfer. account: cap...,Выплата врачам за консультации
6,currency conversion . eur to . rur. type: tran...,Финансовые доходы/расходы
7,currency conversion . eur to . rur. type: tran...,Финансовые доходы/расходы
8,аванс medprestige oü - ronald global pro s.r.o...,Выплата врачам за консультации
9,currency conversion . eur to . rur. type: tran...,Финансовые доходы/расходы


In [12]:
# STEP 4. Label Encoding

from sklearn.preprocessing import LabelEncoder

# Encode text labels into numeric labels
label_encoder = LabelEncoder()

df_model["label"] = label_encoder.fit_transform(df_model[target_col])

# Number of unique classes
num_labels = len(label_encoder.classes_)

print("Number of labels:", num_labels)

# Show label mapping
label_mapping = pd.DataFrame({
    "label_id": range(num_labels),
    "CostElement": label_encoder.classes_
})

display(label_mapping)

Number of labels: 20


,label_id,CostElement
0,0,Аренда офиса
1,1,"Банковские комиссии, платежные платформы"
2,2,"Брендинг, PR"
3,3,ВГО
4,4,Выплата врачам за консультации
5,5,Выплата дежурным врачам
6,6,Доход от медицинских консультаций
7,7,Зарплата административного персонала
8,8,Зарплата с налогами команды R&D
9,9,Зарплата с налогами команды ИТ


In [13]:
# ============================================
# STEP 4.5 Remove duplicates BEFORE split
# ============================================

df_model = df_model.drop_duplicates(subset=["text"]).reset_index(drop=True)

print("After dedup:", df_model.shape)

After dedup: (2710, 6)


In [14]:
# ============================================
# STEP 5. Train / Validation Split + LIMITED Oversampling
# ============================================

from sklearn.model_selection import train_test_split
from sklearn.utils import resample
import pandas as pd

# Split original data first
# Validation must remain natural and untouched
train_df, val_df = train_test_split(
    df_model[["text", "label"]],
    test_size=0.2,
    random_state=42,
    stratify=df_model["label"]
)

print("Original train distribution:")
display(train_df["label"].value_counts().to_frame("count"))

# Limited oversampling
max_count = train_df["label"].value_counts().max()

balanced_parts = []

for label in train_df["label"].unique():
    label_df = train_df[train_df["label"] == label]
    original_count = len(label_df)

    # Do not oversample too aggressively
    target_count = min(max_count, original_count * 5)

    label_resampled = resample(
        label_df,
        replace=True,
        n_samples=target_count,
        random_state=42
    )

    balanced_parts.append(label_resampled)

train_df_balanced = pd.concat(balanced_parts).sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

val_df = val_df.reset_index(drop=True)

print("Balanced train distribution:")
display(train_df_balanced["label"].value_counts().to_frame("count"))

print("Train original shape:", train_df.shape)
print("Train balanced shape:", train_df_balanced.shape)
print("Validation shape:", val_df.shape)

Original train distribution:


,count
label,
1,757
6,564
4,292
3,271
16,47
7,43
11,36
5,36
13,21


Balanced train distribution:


,count
label,
1,757
4,757
3,757
6,757
16,235
7,215
11,180
5,180
13,105


Train original shape: (2168, 2)
Train balanced shape: (4448, 2)
Validation shape: (542, 2)


In [15]:
# ============================================
# STEP 6. Convert pandas DataFrames to Hugging Face Dataset
# ============================================

from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df_balanced)
val_dataset = Dataset.from_pandas(val_df)

print(train_dataset)
print(val_dataset)

Dataset({
    features: ['text', 'label'],
    num_rows: 4448
})
Dataset({
    features: ['text', 'label'],
    num_rows: 542
})


In [16]:
# ============================================
# STEP 6.1. Check train-validation overlap
# ============================================

train_texts = set(train_df_balanced["text"])
val_texts = set(val_df["text"])

overlap = train_texts.intersection(val_texts)

print("Overlap:", len(overlap))

Overlap: 0


In [17]:
# STEP 7. Load tokenizer

from transformers import AutoTokenizer

model_name = "bert-base-multilingual-cased"

# Load tokenizer for multilingual BERT
tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [18]:
# STEP 8. Tokenize datasets

# This function converts text into token IDs that BERT can process
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

# Apply tokenization to train and validation datasets
train_tokenized = train_dataset.map(tokenize_function, batched=True)
val_tokenized = val_dataset.map(tokenize_function, batched=True)

print(train_tokenized)
print(val_tokenized)

Map:   0%|          | 0/4448 [00:00<?, ? examples/s]

Map:   0%|          | 0/542 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 4448
})
Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 542
})


In [19]:
# STEP 9. Load base model

from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

# Load multilingual BERT for classification
base_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [20]:
# STEP 10. Apply LoRA with PEFT

from peft import LoraConfig, get_peft_model, TaskType

# LoRA configuration for BERT sequence classification
# LoRA
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query", "value"]
)

# Add LoRA adapters to the base model
model = get_peft_model(base_model, lora_config)

# Show trainable parameters
model.print_trainable_parameters()

trainable params: 605,204 || all params: 178,474,024 || trainable%: 0.3391


In [21]:
# ============================================
# STEP 11. Define metrics
# ============================================

import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    # Get model predictions and true labels
    logits, labels = eval_pred

    # Convert logits to predicted class IDs
    predictions = np.argmax(logits, axis=-1)

    # Calculate metrics
    acc = accuracy.compute(predictions=predictions, references=labels)
    f1_macro = f1.compute(predictions=predictions, references=labels, average="macro")

    return {
        "accuracy": acc["accuracy"],
        "f1_macro": f1_macro["f1"]
    }

In [22]:
# ============================================
# STEP 12. Training arguments
# ============================================

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./bank_bert_lora_results_v2",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none"
)

In [23]:
# ============================================
# STEP 13. Create Trainer (fixed)
# ============================================

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics
)

print("Trainer is ready.")

Trainer is ready.


In [24]:
print("train_df:", len(train_df))
print("train_df_balanced:", len(train_df_balanced))
print("train_dataset:", len(train_dataset))
print("val_df:", len(val_df))
print("val_dataset:", len(val_dataset))

train_df: 2168
train_df_balanced: 4448
train_dataset: 4448
val_df: 542
val_dataset: 542


In [25]:
train_texts = set(train_df_balanced["text"])
val_texts = set(val_df["text"])

overlap = train_texts.intersection(val_texts)

print("Overlap:", len(overlap))

Overlap: 0


In [26]:
# ============================================
# STEP 14. Train the LoRA model
# ============================================

# Start fine-tuning
train_result = trainer.train()

# Show training summary
print(train_result)

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.445906,0.826941,0.832103,0.201945
2,1.088653,0.588782,0.850554,0.218504
3,0.920085,0.532503,0.869004,0.250451
4,0.787569,0.506224,0.880074,0.285259
5,0.906649,0.501182,0.876384,0.260938


TrainOutput(global_step=1390, training_loss=1.1335605484118565, metrics={'train_runtime': 809.4612, 'train_samples_per_second': 27.475, 'train_steps_per_second': 1.717, 'total_flos': 2946941943152640.0, 'train_loss': 1.1335605484118565, 'epoch': 5.0})


In [27]:
# ============================================
# STEP 15. Evaluate the trained model
# ============================================

eval_result = trainer.evaluate()

print(eval_result)

{'eval_loss': 0.5062236189842224, 'eval_accuracy': 0.8800738007380073, 'eval_f1_macro': 0.2852589686585276, 'eval_runtime': 9.0376, 'eval_samples_per_second': 59.972, 'eval_steps_per_second': 3.762, 'epoch': 5.0}
